# Instrument the 2026 World Cup app with Enprompta

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enprompta/worldcup2026/blob/main/notebooks/enprompta_quickstart.ipynb)

[enprompta.com](https://enprompta.com) &nbsp;|&nbsp; Docs &nbsp;|&nbsp; Community

This notebook takes the **uninstrumented** [`worldcup2026`](https://github.com/Enprompta/worldcup2026)
app and adds Enprompta's three pillars around its single LLM call site:

1. **Tracing** — OpenTelemetry spans for every Claude call, exported to Enprompta.
2. **Prompt registry** — serve the system prompt from Enprompta at runtime instead of hardcoding it.
3. **Evals** — score answers (rule-based + LLM-as-judge) on the traces you just produced.

You'll clone the repo, point it at your Enprompta project, ask a few World Cup
questions, and watch the traces and eval scores appear in your dashboard.

## What you need

- An **Anthropic API key** (`sk-ant-...`)
- Your **Enprompta API key** and **OTLP endpoint** (Project → Settings → Ingest)

> In Colab, add these under the **🔑 Secrets** panel (left sidebar) as
> `ANTHROPIC_API_KEY`, `ENPROMPTA_API_KEY`, and `ENPROMPTA_OTLP_ENDPOINT` so you
> never paste keys into a cell. The next cell falls back to a prompt if a secret
> isn't set.

## Step 1 — Install dependencies

In [ ]:
# Anthropic SDK + OpenTelemetry + the OpenInference auto-instrumentor for Anthropic.
# These are vendor-neutral: Enprompta is an OTLP endpoint, so no proprietary client is required.
!pip -q install \
    "anthropic>=0.116.0" \
    openinference-instrumentation-anthropic \
    opentelemetry-sdk \
    opentelemetry-exporter-otlp-proto-http

# If you publish an Enprompta Python SDK, add it here and use it in place of the raw
# OTLP config below, e.g.:
# !pip -q install enprompta

## Step 2 — Keys and endpoint

In [ ]:
import os
from getpass import getpass

def secret(name, prompt):
    # Prefer Colab Secrets, then existing env var, then interactive prompt.
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name) or getpass(prompt)

os.environ["ANTHROPIC_API_KEY"]      = secret("ANTHROPIC_API_KEY", "🔑 Anthropic API key: ")
ENPROMPTA_API_KEY                    = secret("ENPROMPTA_API_KEY", "🔑 Enprompta API key: ")
ENPROMPTA_OTLP_ENDPOINT              = secret("ENPROMPTA_OTLP_ENDPOINT", "🔑 Enprompta OTLP endpoint (e.g. https://otel.enprompta.com/v1/traces): ")
PROJECT_NAME                         = "worldcup2026-colab"

## Step 3 — Get the app code

We clone the repo and import its shared config (`MODEL`, `SYSTEM_PROMPT`,
`TOOLS`, `extract_citations`) rather than copy-pasting it — so this notebook
stays in sync with the app.

In [ ]:
import sys, os

if not os.path.exists("worldcup2026"):
    !git clone -q https://github.com/Enprompta/worldcup2026.git
sys.path.insert(0, "worldcup2026")

from worldcup import MODEL, SYSTEM_PROMPT, TOOLS, extract_citations
print("Imported call-site config. Model:", MODEL)

## Step 4 — Pillar 1: Tracing

Wire OpenTelemetry to your Enprompta project and turn on the Anthropic
auto-instrumentor. From here, **every** Claude call in this session is captured
as a span (prompt, completion, tokens, latency, tool use) and shipped to
Enprompta — no changes to the app's call site required.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from openinference.instrumentation.anthropic import AnthropicInstrumentor

provider = TracerProvider(resource=Resource.create({"service.name": PROJECT_NAME}))
exporter = OTLPSpanExporter(
    endpoint=ENPROMPTA_OTLP_ENDPOINT,
    headers={"Authorization": f"Bearer {ENPROMPTA_API_KEY}"},  # match your Enprompta ingest auth
)
provider.add_span_processor(BatchSpanProcessor(exporter))
trace.set_tracer_provider(provider)

AnthropicInstrumentor().instrument(tracer_provider=provider)
print("Tracing on →", ENPROMPTA_OTLP_ENDPOINT)

## Step 5 — Pillar 3: Prompt registry (optional but recommended)

Instead of the system prompt baked into `worldcup.py`, fetch a **versioned**
prompt from Enprompta at runtime. Swap the placeholder below for your registry
call; the fallback keeps the notebook runnable before your registry is wired up.

In [ ]:
def get_system_prompt():
    # TODO: replace with your Enprompta prompt-registry client, e.g.
    #   from enprompta import registry
    #   return registry.get("worldcup-system", label="production").render()
    return SYSTEM_PROMPT  # fallback: the prompt shipped in the repo

ACTIVE_SYSTEM_PROMPT = get_system_prompt()
print(ACTIVE_SYSTEM_PROMPT[:160], "...")

## Step 6 — Ask a question (traced)

A notebook-friendly version of the app's `ask()` — same model, tools, and
citation extraction, but it returns the text instead of streaming to a
terminal. Because the Anthropic SDK is instrumented, this call shows up in
Enprompta automatically.

In [ ]:
import anthropic
client = anthropic.Anthropic()

def ask(question, history=None):
    messages = (history or []) + [{"role": "user", "content": question}]
    citations = []
    for _ in range(4):  # cap pause_turn continuations (web_search)
        resp = client.messages.create(
            model=MODEL, max_tokens=4096,
            system=ACTIVE_SYSTEM_PROMPT, tools=TOOLS, messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})
        citations += extract_citations(resp.content)
        if resp.stop_reason == "pause_turn":
            messages.append({"role": "user", "content": "Please continue."})
            continue
        break
    text = "".join(b.text for b in resp.content if getattr(b, "type", None) == "text")
    return text, citations, messages

answer, cites, history = ask("Which cities are hosting the 2026 final and semi-finals?")
print(answer)
print("\nSources:", [c["url"] for c in cites])

## Step 7 — Pillar 2: Evals

Score the answer you just produced. Two cheap checks to start:

- a **rule-based** assertion (did it cite at least one source?), and
- an **LLM-as-judge** relevance score.

In Enprompta these run in CI and on live traffic; here you run them inline so you
can see how a score attaches to a trace.

In [ ]:
# Rule-based
grounded = len(cites) > 0
print("grounded (has_citation):", grounded)

# LLM-as-judge (binary relevance)
JUDGE = ("You are grading a 2026 World Cup assistant. Answer PASS if the response "
         "directly and correctly addresses the question, else FAIL. Reply with one word.")
verdict = client.messages.create(
    model=MODEL, max_tokens=5,
    system=JUDGE,
    messages=[{"role": "user", "content": f"Q: {history[0]['content']}\n\nA: {answer}\n\nGrade:"}],
)
label = "".join(b.text for b in verdict.content if getattr(b, "type", None) == "text").strip().upper()
print("relevance (llm_judge):", label)

# TODO: log {grounded, label} back to the trace via your Enprompta evals client so
# scores sit alongside the span from Step 6.

## Next steps

- Open your **Enprompta project** → you should see the spans from Steps 6–7 with
  token counts, latency, tool calls, and the eval scores.
- Change `ACTIVE_SYSTEM_PROMPT` (Step 5) and re-ask — compare versions in the
  registry and diff the eval scores.
- Add your own eval to Step 7 (faithfulness, tone, no-hallucination) and run it
  over a batch of questions.

The app ships **uninstrumented on purpose** — everything above lives in this
notebook, so learners see exactly what instrumentation adds and where.